In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage
from scipy import ndimage

In [ ]:
filename = 'Leopard'
raw_dimension = 1000

In [ ]:
halfthroat_img = np.fromfile(f"../raw_data/{filename}_halfthroats.raw", dtype=np.int32, sep="")
halfthroat_img = halfthroat_img.reshape([raw_dimension, raw_dimension, raw_dimension])
pores_img = np.fromfile(f"../raw_data/{filename}_pores.raw", dtype=np.int32, sep="")
pores_img = pores_img.reshape([raw_dimension, raw_dimension, raw_dimension])

In [ ]:
pores_boundaries_numbers = np.load(f'../saved_dicts_and_lists/{filename}_pores_boundaries_numbers.npy', allow_pickle=True).item()
pores_boundaries_numbers = {i: pores_boundaries_numbers[i] for i in pores_boundaries_numbers.keys() if pores_boundaries_numbers[i] >= 0}

In [ ]:
pores_neighbours = {i[0]:[] for i in pores_boundaries_numbers.keys()}

In [ ]:
%%time
halfthroat_img_boundaries = np.zeros(halfthroat_img.shape)
cnt = 1
halfthroat_boundaries_numbers = {}

for i, j, k in np.transpose(np.where(skimage.segmentation.find_boundaries(halfthroat_img, connectivity=1, mode='inner', background=0))):
    directions = [
        (1, 0, 0), (-1, 0, 0),
        (0, 1, 0), (0, -1, 0),
        (0, 0, 1), (0, 0, -1)
    ]
    for dx, dy, dz in directions:
        ni, nj, nk = i + dx, j + dy, k + dz
        # Check that the new coordinates are within the array bounds
        if 0 <= ni < raw_dimension and 0 <= nj < raw_dimension and 0 <= nk < raw_dimension:
            if halfthroat_img[i, j, k] != halfthroat_img[ni, nj, nk] and halfthroat_img[ni, nj, nk] != 0:
                pore_numbers_pair = (halfthroat_img[i, j, k], pores_img[i, j, k])
                if pore_numbers_pair not in halfthroat_boundaries_numbers.keys(): # if the pair halfthroat_number - pore_number is absent, add it.
                    halfthroat_boundaries_numbers[pore_numbers_pair] = cnt
                    cnt += 1
                halfthroat_img_boundaries[i, j, k] = halfthroat_boundaries_numbers[pore_numbers_pair]
        

In [ ]:
check_set = {(pores_boundaries_numbers[i], i[0]) for i in pores_boundaries_numbers.keys()}.union({(pores_boundaries_numbers[i], i[1]) for i in pores_boundaries_numbers.keys()})

In [ ]:
my_set = set(halfthroat_boundaries_numbers.keys())

If only one neighbour is found for a halfthroat, this halfthroat is a dead end and can be ignored, it is not important

If a halfthroat ended up in the wrong pore, this is a halfthroat generation error

In [ ]:
plt.imshow(halfthroat_img_boundaries[0])

In [ ]:
if len(my_set.difference(check_set)) == 0:
    print('No unexpected boundaries between halfthroats were added')
else:
    print(f'{len(my_set.difference(check_set))} new boundaries between halfthroats were formed')
    print(f'This is {len(my_set.difference(check_set)) / len(my_set) * 100:.2f}% of all {len(my_set)} halfthroat boundaries')

In [ ]:
halfthroat_img_boundaries.astype(np.int32).tofile(f'../raw_data/{filename}_halfthroats_boundaries.raw')

In [ ]:
np.save(f'../saved_dicts_and_lists/{filename}_halfthroat_boundaries_numbers.npy', halfthroat_boundaries_numbers)